In [38]:
# Import the relevant classes.
from Chapter2.CreateDataset import CreateDataset
from util.VisualizeDataset import VisualizeDataset
from util import util
from pathlib import Path
import pandas as pd
import copy
import os
import sys
import numpy as np

In [23]:
# Chapter 2: Initial exploration of the dataset.
directory = Path('./datasets/bouldering/participant1')

for i, folder in enumerate(directory.iterdir()):
    if folder.is_dir():
        print(f"Folder: {folder}")

    DATASET_PATH = Path(folder)
    RESULT_PATH = Path('./intermediate_datafiles_bouldering/')
    RESULT_FNAME = f'{folder.name}.csv'

    [path.mkdir(exist_ok=True, parents=True) for path in [DATASET_PATH, RESULT_PATH]]
    dataset = CreateDataset(DATASET_PATH, 250)

    # Replace with the new methods and specify unit='s'
    dataset.add_numerical_dataset_with_unit('Accelerometer.csv', "Time (s)", ["X (m/s^2)", "Y (m/s^2)", "Z (m/s^2)"],
                                            'avg', 'acc_', timestamp_unit='s')
    dataset.add_numerical_dataset_with_unit('Gyroscope.csv', "Time (s)", ["X (rad/s)", "Y (rad/s)", "Z (rad/s)"], 'avg',
                                            'gyr_', timestamp_unit='s')

    dataset.add_numerical_dataset_with_unit('Magnetometer.csv', "Time (s)", ["X (µT)", "Y (µT)", "Z (µT)"], 'avg',
                                            'mag_', timestamp_unit='s')
    dataset.add_numerical_dataset_with_unit('Barometer.csv', "Time (s)", ["X (hPa)"], 'avg', 'press_',
                                            timestamp_unit='s')

    result = RESULT_PATH / RESULT_FNAME
    dataset.data_table.to_csv(result, index=False)

Folder: datasets/bouldering/participant1/Easy1 2025-06-03 20-24-57
Reading data from Accelerometer.csv
Reading data from Gyroscope.csv
Reading data from Magnetometer.csv
Reading data from Barometer.csv
Folder: datasets/bouldering/participant1/Hard1 2025-06-09 13-12-42
Reading data from Accelerometer.csv
Reading data from Gyroscope.csv
Reading data from Magnetometer.csv
Reading data from Barometer.csv
Folder: datasets/bouldering/participant1/Hard1 2025-06-09 13-26-33
Reading data from Accelerometer.csv
Reading data from Gyroscope.csv
Reading data from Magnetometer.csv
Reading data from Barometer.csv
Folder: datasets/bouldering/participant1/Hard1 2025-06-09 13-46-36
Reading data from Accelerometer.csv
Reading data from Gyroscope.csv
Reading data from Magnetometer.csv
Reading data from Barometer.csv
Folder: datasets/bouldering/participant1/Medium1 2025-06-09 13-43-08
Reading data from Accelerometer.csv
Reading data from Gyroscope.csv
Reading data from Magnetometer.csv
Reading data from Ba

In [40]:
from scipy.stats import skew, kurtosis

def extract_features(df: pd.DataFrame) -> dict:
    features = {}
    for col in df.columns:
        x = df[col].dropna().values
        features[f'{col}_mean'] = np.mean(x)
        features[f'{col}_std'] = np.std(x)
        features[f'{col}_min'] = np.min(x)
        features[f'{col}_max'] = np.max(x)
        features[f'{col}_skew'] = skew(x)
        features[f'{col}_kurt'] = kurtosis(x)
    return features

In [49]:
from scipy.fft import rfft, rfftfreq

def fourier_features(ts: np.ndarray, sampling_rate: float, top_n=3):
    N = len(ts)
    yf = np.abs(rfft(ts - np.mean(ts)))  # Real FFT, remove DC offset
    xf = rfftfreq(N, 1 / sampling_rate)

    # Ignore the zero-frequency component (DC)
    yf[0] = 0

    # Get top_n peaks
    top_indices = np.argsort(yf)[-top_n:][::-1]
    features = {}
    for i, idx in enumerate(top_indices):
        features[f'freq_{i+1}_Hz'] = xf[idx]
        features[f'freq_{i+1}_amp'] = yf[idx]

    features['spectral_energy'] = np.sum(yf**2)
    return features

In [50]:
X_list = []
y_list = []

sampling_rate = 250  # Adjust to your actual Hz if needed

for file in sorted(Path("intermediate_datafiles_bouldering").glob("*.csv")):
    df = pd.read_csv(file)

    row_features = {}
    for col in df.columns:
        if not np.issubdtype(df[col].dtype, np.number):
            continue
        try:
            feats = fourier_features(df[col].dropna().values, sampling_rate)
            for k, v in feats.items():
                row_features[f"{col}_{k}"] = v
        except Exception as e:
            print(f"Error with {file.name} column {col}: {e}")

    # Label extraction
    name = file.stem.lower()
    if "easy" in name:
        label = "easy"
    elif "medium" in name:
        label = "medium"
    elif "hard" in name:
        label = "hard"
    else:
        continue

    X_list.append(row_features)
    y_list.append(label)


In [51]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = pd.DataFrame(X_list).fillna(0)
y = pd.Series(y_list)

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

print(classification_report(y_test, clf.predict(X_test)))


              precision    recall  f1-score   support

        easy       0.00      0.00      0.00         1
        hard       0.50      0.50      0.50         2
      medium       0.00      0.00      0.00         1

    accuracy                           0.25         4
   macro avg       0.17      0.17      0.17         4
weighted avg       0.25      0.25      0.25         4



/home/aronf/anaconda3/envs/myenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1272: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [54]:
from sklearn.model_selection import cross_val_score
clf = RandomForestClassifier(n_estimators=100, random_state=42)

scores = cross_val_score(clf, X, y, cv=4, scoring='f1_macro')  # better than train/test split on tiny data
print("Cross-validated F1 macro:", scores.mean())

# You can also try class weighting to deal with imbalance:
clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)

print(classification_report(y_test, clf.predict(X_test)))

Cross-validated F1 macro: 0.625
              precision    recall  f1-score   support

        easy       0.00      0.00      0.00         1
        hard       0.50      0.50      0.50         2
      medium       0.00      0.00      0.00         1

    accuracy                           0.25         4
   macro avg       0.17      0.17      0.17         4
weighted avg       0.25      0.25      0.25         4



/home/aronf/anaconda3/envs/myenv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1272: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
